In [1]:
import cv2
import os
import csv
import numpy as np
import pandas as pd
import mediapipe as mp
from scipy.spatial import distance as dist
from skimage.feature import local_binary_pattern
import matplotlib.pyplot as plt



2025-10-02 14:12:01.935442: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-02 14:12:01.945578: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-02 14:12:02.054531: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-02 14:12:02.216148: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-02 14:12:02.380044: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

In [4]:
# ------------------------------
# File paths
# ------------------------------
ear_csv = "ear_features_fullface.csv"
mar_csv = "mar_features_fullface.csv"
lbp_csv = "lbp_features_fullface.csv"

# ------------------------------
# Read CSVs
# ------------------------------
df_ear = pd.read_csv(ear_csv)
df_mar = pd.read_csv(mar_csv)
df_lbp = pd.read_csv(lbp_csv)

print("EAR CSV shape:", df_ear.shape)
print("MAR CSV shape:", df_mar.shape)
print("LBP CSV shape:", df_lbp.shape)

# ------------------------------
# Merge EAR and MAR on 'filename' + 'label'
# ------------------------------
df_merge = pd.merge(df_ear, df_mar, on=['filename', 'label'], how='outer')
print("After merging EAR & MAR:", df_merge.shape)

# Merge the result with LBP
df_merge = pd.merge(df_merge, df_lbp, on=['filename', 'label'], how='outer')
print("After merging with LBP:", df_merge.shape)

# ------------------------------
# Check for missing values
# ------------------------------
print("\nMissing values per column:\n", df_merge.isna().sum())

# ------------------------------
# Drop rows with any NaN (face not detected)
# ------------------------------
df_clean = df_merge.dropna()
df_clean.reset_index(drop=True, inplace=True)

# ------------------------------
# Save final cleaned dataset
# ------------------------------
cleaned_csv = "merged_ml_dataset.csv"
df_clean.to_csv(cleaned_csv, index=False)

# ------------------------------
# Print summary
# ------------------------------
print(f"\nOriginal rows: {len(df_merge)}")
print(f"Cleaned rows (NaN removed): {len(df_clean)}")
print(f"Dropped rows due to NaN: {len(df_merge) - len(df_clean)}")
print(f"Cleaned dataset saved to: {cleaned_csv}")


EAR CSV shape: (66521, 3)
MAR CSV shape: (66521, 3)
LBP CSV shape: (66521, 179)
After merging EAR & MAR: (66521, 4)
After merging with LBP: (66521, 181)

Missing values per column:
 filename      0
label         0
avg_EAR     662
MAR         662
LBP_0       662
           ... 
LBP_172     662
LBP_173     662
LBP_174     662
LBP_175     662
LBP_176     662
Length: 181, dtype: int64

Original rows: 66521
Cleaned rows (NaN removed): 65859
Dropped rows due to NaN: 662
Cleaned dataset saved to: merged_ml_dataset.csv
